# SBATCH Script Generation

Generate SBATCH scripts from configuration for Isambard Slurm jobs.

In [ ]:
#|default_exp sbatch

In [ ]:
#|export
from dataclasses import dataclass, field
from isambard_utils.config import IsambardConfig
from isambard_utils.ssh import _get_config

In [ ]:
#|export
@dataclass
class SbatchConfig:
    """Configuration for generating an SBATCH script."""
    job_name: str
    partition: str = "workq"
    gpus: int = 1
    cpus_per_task: int = 16
    mem: str = "80G"
    time: str = "12:00:00"
    array: str | None = None
    modules: list[str] = field(default_factory=lambda: ["cudatoolkit/24.11_12.6"])
    env_vars: dict[str, str] = field(default_factory=dict)
    pre_commands: list[str] = field(default_factory=list)
    python_script: str | None = None
    python_command: str | None = None
    command: str | None = None

In [ ]:
#|export
def generate(sbatch_config: SbatchConfig, *,
             isambard_config: IsambardConfig | None = None) -> str:
    """Generate a complete SBATCH script string.

    Args:
        sbatch_config: Job-specific configuration.
        isambard_config: Isambard cluster configuration.
    """
    isambard_config = _get_config(isambard_config)
    sc = sbatch_config
    ic = isambard_config

    lines = ["#!/bin/bash"]

    # SBATCH directives
    lines.append(f"#SBATCH --job-name={sc.job_name}")
    lines.append(f"#SBATCH --partition={sc.partition}")
    lines.append(f"#SBATCH --output={ic.logs_dir}/%x_%j.out")
    lines.append(f"#SBATCH --error={ic.logs_dir}/%x_%j.err")
    lines.append(f"#SBATCH --gpus={sc.gpus}")
    lines.append(f"#SBATCH --cpus-per-task={sc.cpus_per_task}")
    lines.append(f"#SBATCH --mem={sc.mem}")
    lines.append(f"#SBATCH --time={sc.time}")
    lines.append(f"#SBATCH --ntasks=1")
    lines.append(f"#SBATCH --chdir={ic.project_dir}")
    if sc.array:
        lines.append(f"#SBATCH --array={sc.array}")
    lines.append("")

    # Shell setup
    lines.append("set -euo pipefail")
    lines.append("")

    # Module loading
    lines.append("module purge")
    for mod in sc.modules:
        lines.append(f"module load {mod}")
    lines.append("")

    # Activate venv
    lines.append("source .venv/bin/activate")
    # Flush Lustre client metadata cache so compute nodes see fresh inodes
    # after packages were reinstalled on the login node
    lines.append("lfs flushctx 2>/dev/null || true")
    lines.append("")

    # Environment variables
    # HF_HOME must point to a shared Lustre path so that ALL HuggingFace caches
    # (hub, modules, etc.) are accessible from compute nodes. Without this,
    # dynamic modules (trust_remote_code) default to ~/.cache/huggingface/modules/
    # which compute nodes may not be able to reach.
    lines.append(f'export HF_HOME="{ic.project_dir}/.hf_home"')
    lines.append(f'export HF_HUB_CACHE="{ic.hf_cache_dir}"')
    lines.append('export HF_HUB_OFFLINE=1')
    lines.append('export HF_HUB_DISABLE_TELEMETRY=1')
    lines.append('export TOKENIZERS_PARALLELISM=false')
    # Force GCC for triton JIT compilation. nvc (from cudatoolkit module)
    # doesn't support GCC-specific flags like -Wno-psabi
    lines.append('export CC=gcc')
    for key, val in sc.env_vars.items():
        lines.append(f'export {key}="{val}"')
    lines.append("")

    # Pre-commands
    for cmd in sc.pre_commands:
        lines.append(cmd)
    if sc.pre_commands:
        lines.append("")

    # Main execution
    if sc.python_script:
        lines.append(f'srun python "{sc.python_script}"')
    elif sc.python_command:
        lines.append(f'srun python -c {_shell_quote(sc.python_command)}')
    elif sc.command:
        lines.append(f'srun {sc.command}')
    else:
        lines.append("# No command specified")

    lines.append("")
    return "\n".join(lines)

In [ ]:
#|export
def _shell_quote(s: str) -> str:
    """Quote a string for safe shell use."""
    import shlex
    return shlex.quote(s)